# 🔥 FireST-GraphNet v2: 补全辅助数据 + 训练

**目标**：在 Google Drive 已有 VIIRS 数据集（`af_*`/`ba_*`/`pred_*`）的基础上，
自动补全缺失的 FirePred 辅助数据（`firepred_*.npy`）和 LULC 土地覆盖数据（`lulc_*.npy`），
然后进行完整训练。

**两种策略**：
- **方案 A（快速，~5 分钟）**：如果 `pred_*_img` 恰好包含 19 个 FirePred 频段，直接复制为 `firepred_*.npy`
- **方案 B（完整重建，~30–90 分钟）**：从原始 GeoTIFF 重新生成所有数据（包括 `af_*`），确保对齐

**流程**：
1. ✅ 环境设置 + 挂载 Drive
2. ✅ 扫描 Drive，确认缺失文件
3. ✅ 生成缺失的辅助数据（自动选择最优策略）
4. ✅ 复制所有数据到 Colab
5. ✅ 验证数据完整性和对齐
6. ✅ 安装训练依赖
7. ✅ 训练模型
8. ✅ 保存结果到 Google Drive
9. ✅ 可视化训练曲线

**Drive 数据路径**：
```
My Drive/FireST-GraphNet/
├── FireST-GraphNet-Clean/
│   ├── firest_graphnet/          # 模型代码
│   ├── config_run12_1.yaml       # 训练配置
│   ├── dataset_train/            # 训练集
│   └── dataset_val/              # 验证集
└── ts-satfire_raw/               # 原始 GeoTIFF（方案 B 需要）
```

## 📋 准备工作

确保 Google Drive 中包含：

| 文件 | 路径 | 状态 |
|------|------|------|
| VIIRS 训练集 | `dataset_train/af_train_img_*.npy` | ✅ 已有 |
| VIIRS 标签 | `dataset_train/af_train_label_*.npy` | ✅ 已有 |
| BA 训练集 | `dataset_train/ba_train_img_*.npy` | ✅ 已有 |
| Pred 训练集 | `dataset_train/pred_train_img_*.npy` | ✅ 已有（方案 A 需要）|
| FirePred 辅助 | `dataset_train/firepred_train.npy` | ❌ **缺失 → 将生成** |
| LULC 土地覆盖 | `dataset_train/lulc_train.npy` | ❌ **缺失 → 将生成** |
| 原始 GeoTIFF | `ts-satfire_raw/` (192 events) | 方案 B 需要 |

---

## 🚀 Step 1: 环境设置

In [ ]:
# 检查 GPU 和磁盘空间
!nvidia-smi

import os
import sys
import shutil
from pathlib import Path

print("\n=== Colab 环境信息 ===")
print(f"可用磁盘空间: {shutil.disk_usage('/').free / (1024**3):.1f} GB")
print(f"工作目录: {os.getcwd()}")

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 设置项目路径
PROJECT_PATH = "/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean"
RAW_DATA_PATH = "/content/drive/MyDrive/FireST-GraphNet/ts-satfire_raw"

DRIVE_TRAIN = f"{PROJECT_PATH}/dataset_train"
DRIVE_VAL   = f"{PROJECT_PATH}/dataset_val"

# 验证路径
assert Path(PROJECT_PATH).exists(), f"❌ 未找到项目文件夹: {PROJECT_PATH}"
print(f"✅ 项目路径: {PROJECT_PATH}")
print(f"✅ 训练集目录: {DRIVE_TRAIN} (exists={Path(DRIVE_TRAIN).exists()})")
print(f"✅ 验证集目录: {DRIVE_VAL} (exists={Path(DRIVE_VAL).exists()})")
print(f"   原始数据路径: {RAW_DATA_PATH} (exists={Path(RAW_DATA_PATH).exists()})")

## 🔍 Step 2: 扫描现有数据

In [ ]:
import numpy as np

print("=" * 60)
print("📂 扫描 Google Drive 数据文件")
print("=" * 60 + "\n")

# 需要检查的文件
check_files = {
    "train": {
        "af_img":      f"{DRIVE_TRAIN}/af_train_img_seqtoseq_alll_1i_1.npy",
        "af_label":    f"{DRIVE_TRAIN}/af_train_label_seqtoseq_alll_1i_1.npy",
        "ba_img":      f"{DRIVE_TRAIN}/ba_train_img_seqtoseq_alll_1i_1.npy",
        "ba_label":    f"{DRIVE_TRAIN}/ba_train_label_seqtoseq_alll_1i_1.npy",
        "pred_img":    f"{DRIVE_TRAIN}/pred_train_img_seqtoseq_alll_1i_1.npy",
        "pred_label":  f"{DRIVE_TRAIN}/pred_train_label_seqtoseq_alll_1i_1.npy",
        "firepred":    f"{DRIVE_TRAIN}/firepred_train.npy",
        "lulc":        f"{DRIVE_TRAIN}/lulc_train.npy",
    },
    "val": {
        "af_img":      f"{DRIVE_VAL}/af_val_img_seqtoseq_alll_1i_1.npy",
        "af_label":    f"{DRIVE_VAL}/af_val_label_seqtoseq_alll_1i_1.npy",
        "ba_img":      f"{DRIVE_VAL}/ba_val_img_seqtoseq_alll_1i_1.npy",
        "ba_label":    f"{DRIVE_VAL}/ba_val_label_seqtoseq_alll_1i_1.npy",
        "pred_img":    f"{DRIVE_VAL}/pred_val_img_seqtoseq_alll_1i_1.npy",
        "pred_label":  f"{DRIVE_VAL}/pred_val_label_seqtoseq_alll_1i_1.npy",
        "firepred":    f"{DRIVE_VAL}/firepred_val.npy",
        "lulc":        f"{DRIVE_VAL}/lulc_val.npy",
    },
}

missing_critical = []  # 训练必需但缺失
missing_aux = []       # 辅助数据缺失（本 notebook 要生成）

for split, files in check_files.items():
    print(f"--- {split.upper()} ---")
    for key, path in files.items():
        p = Path(path)
        if p.exists():
            size_mb = p.stat().st_size / (1024**2)
            try:
                arr = np.load(path, mmap_mode='r')
                print(f"  ✅ {key:12s}  shape={str(arr.shape):30s}  {size_mb:8.1f} MB")
            except Exception:
                print(f"  ✅ {key:12s}  {size_mb:8.1f} MB")
        else:
            print(f"  ❌ {key:12s}  NOT FOUND")
            if key in ("af_img", "af_label"):
                missing_critical.append(f"{split}/{key}")
            if key in ("firepred", "lulc"):
                missing_aux.append(f"{split}/{key}")
    print()

if missing_critical:
    print("⛔ 训练必需文件缺失（af_img, af_label），无法继续！")
    print("   缺失:", missing_critical)
    print("   如采用方案 B 重建，可在下一步自动生成。")

if missing_aux:
    print(f"📌 辅助数据缺失: {missing_aux}")
    print("   将在 Step 3 中自动生成。")
else:
    print("✅ 所有辅助数据已存在，Step 3 将自动跳过。")

## ⚙️ Step 3: 生成缺失的辅助数据

**自动策略选择**：

- **方案 A**：检查 `pred_*_img` 是否包含 19 个 FirePred 频段且样本数与 `af_*_img` 对齐
  - ✅ 直接复制为 `firepred_*.npy`（快速）
  - LULC 使用零值占位（模型可正常运行，仅缺少 1ch 土地覆盖信息）

- **方案 B**：从原始 GeoTIFF 重新生成所有数据
  - ⚠️ 会重新生成 `af_*_img`/`af_*_label`（确保与 firepred/lulc 对齐）
  - 需要 `ts-satfire_raw/` 原始数据（~71GB）
  - 耗时较长（30–90 分钟）

In [ ]:
import numpy as np
import shutil
from pathlib import Path

# === 检查是否需要生成 ===
fp_train_path = Path(DRIVE_TRAIN) / "firepred_train.npy"
fp_val_path   = Path(DRIVE_VAL)   / "firepred_val.npy"
lulc_train_path = Path(DRIVE_TRAIN) / "lulc_train.npy"
lulc_val_path   = Path(DRIVE_VAL)   / "lulc_val.npy"

need_firepred = not (fp_train_path.exists() and fp_val_path.exists())
need_lulc     = not (lulc_train_path.exists() and lulc_val_path.exists())

if not need_firepred and not need_lulc:
    print("✅ firepred_*.npy 和 lulc_*.npy 已存在，跳过生成！")
    STRATEGY_USED = "skip"
else:
    # --- 加载现有 af 数据获取形状信息 ---
    af_train_file = Path(DRIVE_TRAIN) / "af_train_img_seqtoseq_alll_1i_1.npy"
    af_val_file   = Path(DRIVE_VAL)   / "af_val_img_seqtoseq_alll_1i_1.npy"

    af_train_exists = af_train_file.exists()
    af_val_exists   = af_val_file.exists()

    if af_train_exists and af_val_exists:
        af_train = np.load(str(af_train_file), mmap_mode='r')
        af_val   = np.load(str(af_val_file),   mmap_mode='r')
        N_train = af_train.shape[0]
        N_val   = af_val.shape[0]
        H = af_train.shape[3]
        W = af_train.shape[4]
        print(f"现有数据: N_train={N_train}, N_val={N_val}, H={H}, W={W}\n")
    else:
        N_train = N_val = H = W = None
        print("⚠️ af_*_img 文件不存在，将使用方案 B 从头生成所有数据\n")

    # === 方案 A: 尝试用 pred_*_img 作为 firepred ===
    strategy_a = False
    if need_firepred and af_train_exists and af_val_exists:
        pred_train_file = Path(DRIVE_TRAIN) / "pred_train_img_seqtoseq_alll_1i_1.npy"
        pred_val_file   = Path(DRIVE_VAL)   / "pred_val_img_seqtoseq_alll_1i_1.npy"

        if pred_train_file.exists() and pred_val_file.exists():
            pred_train = np.load(str(pred_train_file), mmap_mode='r')
            pred_val   = np.load(str(pred_val_file),   mmap_mode='r')
            print(f"pred_train_img: shape={pred_train.shape}")
            print(f"pred_val_img:   shape={pred_val.shape}")

            # 检查: 19 个频段 + 样本数对齐
            if (pred_train.shape[1] == 19 and pred_val.shape[1] == 19
                and pred_train.shape[0] == N_train
                and pred_val.shape[0] == N_val):
                strategy_a = True
                print("\n✅ 方案 A 可用！pred_*_img 包含 19 个 FirePred 频段且样本数对齐。")
            else:
                print(f"\n⚠️ 方案 A 不适用:")
                if pred_train.shape[1] != 19:
                    print(f"   - 频段数 = {pred_train.shape[1]}（期望 19）")
                if pred_train.shape[0] != N_train:
                    print(f"   - 样本数不对齐: pred={pred_train.shape[0]} vs af={N_train}")
        else:
            print("⚠️ pred_*_img 文件不存在，无法使用方案 A")

    # === 执行生成 ===
    if strategy_a:
        # ─── 方案 A: 复制 pred_img → firepred ───
        STRATEGY_USED = "A"
        print("\n" + "=" * 60)
        print("🚀 执行方案 A: 复制 pred_*_img → firepred_*.npy")
        print("=" * 60 + "\n")

        if need_firepred:
            print("📂 生成 firepred 文件...")
            shutil.copy2(str(pred_train_file), str(fp_train_path))
            size_mb = fp_train_path.stat().st_size / (1024**2)
            print(f"  ✅ firepred_train.npy  ({size_mb:.1f} MB)")

            shutil.copy2(str(pred_val_file), str(fp_val_path))
            size_mb = fp_val_path.stat().st_size / (1024**2)
            print(f"  ✅ firepred_val.npy    ({size_mb:.1f} MB)")

        if need_lulc:
            print("\n📂 生成 LULC 占位文件（零值）...")
            lulc_zeros_train = np.zeros((N_train, 1, H, W), dtype=np.float32)
            np.save(str(lulc_train_path), lulc_zeros_train)
            print(f"  ✅ lulc_train.npy  shape={lulc_zeros_train.shape}")

            lulc_zeros_val = np.zeros((N_val, 1, H, W), dtype=np.float32)
            np.save(str(lulc_val_path), lulc_zeros_val)
            print(f"  ✅ lulc_val.npy    shape={lulc_zeros_val.shape}")
            print("  ⓘ  LULC 为零值占位（28ch 中仅占 1ch，对模型影响很小）")

        print("\n✅ 方案 A 完成！所有辅助数据已保存到 Google Drive。")

    else:
        # ─── 方案 B: 从原始 GeoTIFF 重新生成 ───
        STRATEGY_USED = "B"
        print("\n" + "=" * 60)
        print("🚀 执行方案 B: 从原始 GeoTIFF 重新生成所有数据")
        print("=" * 60)
        print("⚠️  此方案会重新生成 af_*_img/label + firepred + lulc")
        print("    以确保所有文件之间的对齐关系。")
        print("    原有的 ba_*/pred_* 文件不受影响。")
        print("    ⏰ 预计耗时 30–90 分钟\n")

        assert Path(RAW_DATA_PATH).exists(), (
            f"❌ 方案 B 需要原始 GeoTIFF 数据: {RAW_DATA_PATH}\n"
            "请确保 ts-satfire_raw/ 已上传到 Google Drive"
        )

        # 安装 rasterio（读取 GeoTIFF 需要）
        !pip install -q rasterio

        sys.path.insert(0, PROJECT_PATH)
        from firest_graphnet.data.preprocess_raw import (
            discover_event_ids, process_all_events
        )

        raw_dir = Path(RAW_DATA_PATH)
        event_ids = discover_event_ids(raw_dir)
        print(f"发现 {len(event_ids)} 个火灾事件")

        if len(event_ids) == 0:
            raise RuntimeError(
                f"在 {RAW_DATA_PATH} 中未找到有效事件！\n"
                "请确保目录结构为: ts-satfire_raw/<event_id>/VIIRS_Day/"
            )

        # 80/20 train/val split（固定随机种子确保可复现）
        rng = np.random.RandomState(42)
        perm = rng.permutation(len(event_ids))
        n_val = max(1, int(len(event_ids) * 0.2))
        train_ids = [event_ids[i] for i in sorted(perm[n_val:])]
        val_ids   = [event_ids[i] for i in sorted(perm[:n_val])]

        print(f"训练集: {len(train_ids)} 个事件")
        print(f"验证集: {len(val_ids)} 个事件\n")

        # 处理训练集
        print("=" * 60)
        print(f"处理训练集 → {DRIVE_TRAIN}")
        print("=" * 60)
        os.makedirs(DRIVE_TRAIN, exist_ok=True)
        process_all_events(
            raw_dir=raw_dir,
            out_dir=DRIVE_TRAIN,
            event_ids=train_ids,
            mode='train',
            ts_length=1,
            interval=1,
            image_size=256,
        )

        # 处理验证集
        print("\n" + "=" * 60)
        print(f"处理验证集 → {DRIVE_VAL}")
        print("=" * 60)
        os.makedirs(DRIVE_VAL, exist_ok=True)
        process_all_events(
            raw_dir=raw_dir,
            out_dir=DRIVE_VAL,
            event_ids=val_ids,
            mode='val',
            ts_length=1,
            interval=1,
            image_size=256,
        )

        print("\n✅ 方案 B 完成！所有数据已重新生成并保存到 Google Drive。")

print(f"\n使用的策略: {STRATEGY_USED}")

## 📋 Step 4: 复制数据到 Colab 本地

从 Drive 复制训练所需文件到 `/content`（读取更快）。

In [ ]:
import shutil
import numpy as np
from pathlib import Path
import os

# Redefine project paths to ensure they are available in this cell
PROJECT_PATH = "/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean"
RAW_DATA_PATH = "/content/drive/MyDrive/FireST-GraphNet/ts-satfire_raw"
DRIVE_TRAIN = f"{PROJECT_PATH}/dataset_train"
DRIVE_VAL   = f"{PROJECT_PATH}/dataset_val"

DATASET_TRAIN = "/content/dataset_train"
DATASET_VAL   = "/content/dataset_val"

os.makedirs(DATASET_TRAIN, exist_ok=True)
os.makedirs(DATASET_VAL, exist_ok=True)

# 训练所需文件（forecast mode 只需 af + firepred + lulc）
files_to_copy = {
    "train": [
        ("af_train_img_seqtoseq_alll_1i_1.npy",   True),
        ("af_train_label_seqtoseq_alll_1i_1.npy", True),
        ("firepred_train.npy",                      True),
        ("lulc_train.npy",                          False),  # optional
    ],
    "val": [
        ("af_val_img_seqtoseq_alll_1i_1.npy",     True),
        ("af_val_label_seqtoseq_alll_1i_1.npy",   True),
        ("firepred_val.npy",                        True),
        ("lulc_val.npy",                            False),  # optional
    ],
}

print("=" * 60)
print("📂 从 Google Drive 复制数据到 Colab 本地")
print("=" * 60 + "\n")

missing = []
for split, file_list in files_to_copy.items():
    src_dir  = DRIVE_TRAIN if split == "train" else DRIVE_VAL
    dest_dir = DATASET_TRAIN if split == "train" else DATASET_VAL
    print(f"{split.upper()}:")
    for filename, required in file_list:
        src  = Path(src_dir) / filename
        dest = Path(dest_dir) / filename
        if src.exists():
            src_size = src.stat().st_size
            # Re-copy if destination doesn't exist or is a different size (partial copy)
            if not dest.exists() or dest.stat().st_size != src_size:
                if dest.exists():
                    print(f"  ⚠️ {filename} 文件不完整（大小不匹配），重新复制...")
                shutil.copy2(src, dest)
            size_mb = dest.stat().st_size / (1024**2)
            print(f"  ✅ {filename:50s} {size_mb:8.1f} MB")
        elif required:
            print(f"  ❌ {filename:50s} NOT FOUND (required!)")
            missing.append(f"{split}/{filename}")
        else:
            print(f"  ⚠️ {filename:50s} NOT FOUND (optional, will use zeros)")
    print()

if missing:
    raise FileNotFoundError(
        "\n❌ 以下必需文件缺失：\n" +
        "\n".join(f"  - {f}" for f in missing)
    )

print("✅ 数据复制完成！")


## ✅ Step 5: 验证数据完整性和对齐

In [ ]:
import numpy as np
from pathlib import Path

print("=" * 60)
print("🔍 验证数据完整性")
print("=" * 60 + "\n")

all_ok = True

def safe_load(path):
    """Load numpy array, falling back to full load if mmap fails (truncated file)."""
    try:
        return np.load(path, mmap_mode='r')
    except ValueError as e:
        if "mmap length is greater than file size" in str(e):
            size_mb = Path(path).stat().st_size / (1024**2)
            raise RuntimeError(
                f"❌ 文件已损坏（截断）: {path}\n"
                f"   文件大小: {size_mb:.1f} MB，但 numpy 头部声明的数据更大。\n"
                f"   请重新运行 Step 4 重新复制该文件。"
            ) from e
        raise

for split, (data_dir, mode) in {"TRAIN": (DATASET_TRAIN, "train"), "VAL": (DATASET_VAL, "val")}.items():
    print(f"--- {split} ---")

    # AF image + label
    try:
        af_img = safe_load(f"{data_dir}/af_{mode}_img_seqtoseq_alll_1i_1.npy")
        af_lbl = safe_load(f"{data_dir}/af_{mode}_label_seqtoseq_alll_1i_1.npy")
    except RuntimeError as e:
        print(e)
        all_ok = False
        continue

    N = af_img.shape[0]
    print(f"  af_img:   {af_img.shape}")
    print(f"  af_label: {af_lbl.shape}")

    # 样本数一致性
    if af_img.shape[0] != af_lbl.shape[0]:
        print(f"  ❌ af img/label 样本数不一致!")
        all_ok = False
    else:
        print(f"  ✅ af img/label 样本数一致: N={N}")

    # FirePred 对齐
    fp_path = Path(data_dir) / f"firepred_{mode}.npy"
    if fp_path.exists():
        try:
            fp = safe_load(str(fp_path))
            print(f"  firepred: {fp.shape}")
            if fp.shape[0] != N:
                print(f"  ❌ firepred N={fp.shape[0]} != af N={N} — 对齐失败!")
                print(f"     训练时将回退到零值占位（PlaceholderFirePred）")
                all_ok = False
            elif fp.shape[1] != 19:
                print(f"  ⚠️ firepred 频段数={fp.shape[1]}（期望 19）")
            else:
                print(f"  ✅ firepred 对齐: N={fp.shape[0]}, C=19")
        except RuntimeError as e:
            print(e)
            all_ok = False
    else:
        print(f"  ⚠️ firepred 文件不存在，训练时使用零值占位")

    # LULC 对齐
    lulc_path = Path(data_dir) / f"lulc_{mode}.npy"
    if lulc_path.exists():
        try:
            lulc = safe_load(str(lulc_path))
            print(f"  lulc:     {lulc.shape}")
            if lulc.shape[0] != N:
                print(f"  ⚠️ lulc N={lulc.shape[0]} != af N={N}")
            else:
                print(f"  ✅ lulc 对齐: N={lulc.shape[0]}")
        except RuntimeError as e:
            print(e)
            all_ok = False
    else:
        print(f"  ⚠️ lulc 文件不存在，训练时使用零值占位")

    print()

if all_ok:
    print("✅ 数据验证通过！可以开始训练。")
else:
    print("⚠️ 存在对齐问题，模型会使用零值占位继续训练。")
    print("   如需完整数据，请使用方案 B 重新生成。")


## 📦 Step 6: 安装训练依赖

In [ ]:
# 安装 FireST-GraphNet 依赖
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q pyyaml tensorboard scikit-learn matplotlib

print("\n✅ 训练依赖安装完成")

In [ ]:
# 验证 PyTorch 和 CUDA
import torch

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 内存: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

## 🏋️ Step 7: 训练模型

In [ ]:
# 设置训练参数
os.chdir("/content")
sys.path.insert(0, PROJECT_PATH)

# 选择配置文件
CONFIG_FILE = f"{PROJECT_PATH}/config_run22.yaml"

if Path(CONFIG_FILE).exists():
    print(f"✅ 配置文件: {CONFIG_FILE}")
    !cat {CONFIG_FILE}
else:
    print(f"❌ 配置文件不存在: {CONFIG_FILE}")
    # 尝试其他配置文件
    for alt in ["config_run12.yaml", "firest_graphnet/configs/base.yaml"]:
        alt_path = f"{PROJECT_PATH}/{alt}"
        if Path(alt_path).exists():
            CONFIG_FILE = alt_path
            print(f"   使用备选配置: {CONFIG_FILE}")
            !cat {CONFIG_FILE}
            break
    else:
        print("   请上传配置文件到 Google Drive")

# 自动提取 run 名称（如 config_run13.yaml -> run13）
RUN_NAME = Path(CONFIG_FILE).stem.replace("config_", "")
if RUN_NAME == "base":
    RUN_NAME = "run_base"
print(f"\n🔄 当前运行名称 (自动提取): {RUN_NAME}")

In [ ]:
# 修改配置文件中的数据路径
import yaml

with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

# 更新数据路径指向 Colab 本地
config['data']['root'] = "/content"
config['data']['root_train'] = DATASET_TRAIN
config['data']['root_val'] = DATASET_VAL

# 保存临时配置
temp_config = "/content/config_colab_v2.yaml"
with open(temp_config, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Colab 配置文件: {temp_config}")
print(f"\n根数据路径: {config['data']['root']}")
print(f"训练数据路径: {config['data']['root_train']}")
print(f"验证数据路径: {config['data']['root_val']}")

In [ ]:
%%time
# 安装 torch-cluster（从预编译源）
import torch
pt_version = torch.__version__
print(f"当前 PyTorch 版本: {pt_version}")

pyg_wheel_url = f"https://data.pyg.org/whl/torch-{pt_version}.html"
print(f"预编译源: {pyg_wheel_url}\n")

!pip install -q torch-cluster -f {pyg_wheel_url}
print("\n✅ torch-cluster 安装完成！")

In [ ]:
import glob

# 修复可能残留的硬编码 Windows 路径
!sed -i 's|C:\\Users\\22125\\Desktop\\FireST-GraphNet|/content|g' {PROJECT_PATH}/firest_graphnet/data/dataset.py
!sed -i 's|C:\\Users\\22125\\Desktop\\FireST-GraphNet|/content|g' {PROJECT_PATH}/firest_graphnet/scripts/train.py

# ── 修复 BatchNorm train/eval 失配（val IoU=0 根因）─────────────────────────────
# 问题：model.eval() 时 BN 使用 running stats，早期训练这些 stats 未收敛，
#        导致 val 的所有 logit < 0，sigmoid < 0.5，pred_pos_ratio = 0, IoU = 0。
# 修复：val 时 dropout 等层保持 eval，但 BN 层切回 train 模式（使用 batch stats）。
TRAINER_PY = f"{PROJECT_PATH}/firest_graphnet/training/trainer.py"

# 检查是否已修复
import subprocess
result = subprocess.run(["grep", "-c", "BatchNorm1d", TRAINER_PY], capture_output=True, text=True)
already_fixed = int(result.stdout.strip()) > 0

if not already_fixed:
    # 用 Python 直接写入修复（避免 sed 多行问题）
    with open(TRAINER_PY, 'r') as f:
        code = f.read()

    OLD = """        else:
            self.model.eval()"""

    NEW = """        else:
            # Keep BN in train mode during validation so it uses batch statistics
            # (running stats are unreliable early in training and cause val IoU=0).
            self.model.eval()
            for m in self.model.modules():
                if isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
                    m.train()"""

    if OLD in code:
        code = code.replace(OLD, NEW, 1)  # only replace first occurrence (ForecastTrainer)
        with open(TRAINER_PY, 'w') as f:
            f.write(code)
        print("✅ trainer.py: BN eval-mode fix applied")
    else:
        print("⚠️ trainer.py: pattern not found — may already be patched or code changed")
else:
    print("✅ trainer.py: BN fix already present")

# ── 修复 --task 强制覆盖 config pos_weight 的问题 ────────────────────────────────
# 问题：--task af 会把 config 里的 pos_weight 强制改为 5.5，config_run14 里是 7.0
TRAIN_PY = f"{PROJECT_PATH}/firest_graphnet/scripts/train.py"
with open(TRAIN_PY, 'r') as f:
    train_code = f.read()

OLD_TASK = '''        cfg["training"]["pos_weight"] = _TASK_POS_WEIGHT[args.task]
        print(f"[Task] {args.task.upper()}  "
              f"label_channel={_TASK_CHANNEL[args.task]}  "
              f"pos_weight={_TASK_POS_WEIGHT[args.task]}")'''

NEW_TASK = '''        if "pos_weight" not in cfg.get("training", {}):
            cfg["training"]["pos_weight"] = _TASK_POS_WEIGHT[args.task]
        print(f"[Task] {args.task.upper()}  "
              f"label_channel={_TASK_CHANNEL[args.task]}  "
              f"pos_weight={cfg[\'training\'].get(\'pos_weight\', _TASK_POS_WEIGHT[args.task])}")'''

if OLD_TASK in train_code:
    train_code = train_code.replace(OLD_TASK, NEW_TASK, 1)
    with open(TRAIN_PY, 'w') as f:
        f.write(train_code)
    print("✅ train.py: pos_weight override fix applied")
else:
    print("ℹ️  train.py: pos_weight fix already present or pattern changed")

# ── 训练任务选择 ──────────────────────────────────────────────────────────
# 可选值:
#   "af"  → Active Fire 活跃火灾检测 (label_channel=1, pos_weight=5.5) ← 默认
#   "ba"  → Burned Area 过火面积映射 (label_channel=0, pos_weight=2.5)
#   "frp" → Fire Radiative Power      (label_channel=2, pos_weight=5.5)
TASK = "af"
TASK_FLAG = f"--task {TASK}"
print(f"\n训练任务: {TASK.upper()}")

# === 断点续训：从 Drive 恢复最新 checkpoint ===
LOCAL_OUTPUT = f"/content/outputs/{RUN_NAME}"
DRIVE_OUTPUT_CKPT = f"{PROJECT_PATH}/outputs/{RUN_NAME}"
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

RESUME_FLAG = ""

# 优先找 best_model.pt，其次找最新的 checkpoint_epoch*.pt
def _latest_checkpoint(ckpt_dir):
    best = Path(ckpt_dir) / "best_model.pt"
    if best.exists():
        return str(best)
    pattern = str(Path(ckpt_dir) / "checkpoint_epoch*.pt")
    candidates = sorted(glob.glob(pattern))
    return candidates[-1] if candidates else None

# 1. 先看 Colab 本地有没有（可能同一 session 内）
local_ckpt = _latest_checkpoint(LOCAL_OUTPUT)
# 2. 如果没有，从 Drive 复制
if local_ckpt is None:
    drive_ckpt = _latest_checkpoint(DRIVE_OUTPUT_CKPT)
    if drive_ckpt is not None:
        dest = Path(LOCAL_OUTPUT) / Path(drive_ckpt).name
        print(f"📂 从 Drive 恢复 checkpoint: {Path(drive_ckpt).name}")
        shutil.copy2(drive_ckpt, dest)
        # 同时复制 metrics.csv 以延续曲线
        drive_csv = Path(DRIVE_OUTPUT_CKPT) / "metrics.csv"
        if drive_csv.exists():
            shutil.copy2(str(drive_csv), str(Path(LOCAL_OUTPUT) / "metrics.csv"))
        local_ckpt = str(dest)

if local_ckpt:
    RESUME_FLAG = f"--resume {local_ckpt}"
    print(f"✅ 将从 checkpoint 续训: {Path(local_ckpt).name}")
else:
    print("ℹ️  未找到 checkpoint，从头训练。")

# 开始训练（同时输出到屏幕和日志文件）
LOG_FILE = f"{LOCAL_OUTPUT}/train.log"
print(f"\n🏋️ 开始训练...  任务={TASK.upper()}  日志: {LOG_FILE}\n")

!python {PROJECT_PATH}/firest_graphnet/scripts/train.py --config {temp_config} --mode forecast {TASK_FLAG} {RESUME_FLAG} 2>&1 | tee {LOG_FILE}

print("\n✅ 训练完成！")


## 💾 Step 8: 保存结果到 Google Drive

In [ ]:
import pandas as pd

OUTPUT_DIR   = "/content/outputs"
DRIVE_OUTPUT = f"{PROJECT_PATH}/outputs/colab_v2_{RUN_NAME}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"

# 将训练结果保存到 Drive（带时间戳的快照 + 覆盖 run_name 目录供断点续训使用）
if Path(OUTPUT_DIR).exists():
    print(f"💾 保存训练结果到 Google Drive...")

    # 1. 带时间戳的完整快照（归档用）
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f"  ✅ 快照: {DRIVE_OUTPUT}")

    # 2. 覆盖指定 run 目录（断点续训用 —— 保留最新 checkpoint）
    DRIVE_RESUME_DIR = f"{PROJECT_PATH}/outputs/{RUN_NAME}"
    os.makedirs(DRIVE_RESUME_DIR, exist_ok=True)
    for fname in ["best_model.pt", "metrics.csv", "config.yaml", "train.log"]:
        src_f = Path(OUTPUT_DIR) / RUN_NAME / fname
        if src_f.exists():
            shutil.copy2(str(src_f), str(Path(DRIVE_RESUME_DIR) / fname))
    # 也复制所有 checkpoint_epoch*.pt
    for ckpt in Path(OUTPUT_DIR).glob(f"{RUN_NAME}/checkpoint_epoch*.pt"):
        shutil.copy2(str(ckpt), str(Path(DRIVE_RESUME_DIR) / ckpt.name))
    print(f"  ✅ 断点续训目录: {DRIVE_RESUME_DIR}")

    !ls -lh {DRIVE_RESUME_DIR}
else:
    print("⚠️ 未找到输出目录，请确认训练已完成。")

## 📊 Step 9: 可视化训练曲线

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Use LOCAL_OUTPUT which correctly points to /content/outputs/run_name
metrics_file = Path(LOCAL_OUTPUT) / "metrics.csv"

if metrics_file.exists():
    df = pd.read_csv(metrics_file)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Loss
    axes[0, 0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0, 0].plot(df['epoch'], df['val_loss'], label='Val')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # F1 Score
    if 'val_f1' in df.columns:
        axes[0, 1].plot(df['epoch'], df['val_f1'])
        axes[0, 1].set_title('Validation F1 Score')
        axes[0, 1].grid(True)

    # IoU
    if 'val_iou' in df.columns:
        axes[1, 0].plot(df['epoch'], df['val_iou'])
        axes[1, 0].set_title('Validation IoU')
        axes[1, 0].grid(True)

    # Precision & Recall
    if 'val_precision' in df.columns and 'val_recall' in df.columns:
        axes[1, 1].plot(df['epoch'], df['val_precision'], label='Precision')
        axes[1, 1].plot(df['epoch'], df['val_recall'], label='Recall')
        axes[1, 1].set_title('Precision & Recall')
        axes[1, 1].legend()
        axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(f"{LOCAL_OUTPUT}/training_curves.png", dpi=150)
    plt.show()

    print("\n最佳验证指标：")
    if 'val_f1' in df.columns:
        best_idx = df['val_f1'].idxmax()
        print(f"Epoch {df.loc[best_idx, 'epoch']}:")
        print(f"  F1: {df.loc[best_idx, 'val_f1']:.4f}")
        if 'val_iou' in df.columns:
            print(f"  IoU: {df.loc[best_idx, 'val_iou']:.4f}")
else:
    print(f"⚠️ 未找到指标文件: {metrics_file}")

## 🎉 完成！

### 工作流程总结

1. ✅ 环境检查 + 挂载 Google Drive
2. ✅ 扫描 Drive 确认数据状态
3. ✅ 生成缺失的辅助数据（firepred + lulc）
4. ✅ 复制所有数据到 Colab
5. ✅ 验证数据完整性和对齐
6. ✅ 安装训练依赖
7. ✅ 训练模型
8. ✅ 保存结果到 Google Drive
9. ✅ 可视化训练曲线

---

### 策略说明

| 策略 | 条件 | 耗时 | 说明 |
|------|------|------|------|
| A（快速） | `pred_*_img` 有 19 频段且 N 对齐 | ~5 分钟 | 复制为 firepred，LULC 用零值 |
| B（完整） | A 不适用，或需要完整数据 | 30–90 分钟 | 从 GeoTIFF 重建，含真实 LULC |
| 跳过 | firepred/lulc 已存在 | 0 分钟 | 直接进入训练 |

### 再次运行

生成的 `firepred_*.npy` 和 `lulc_*.npy` 已保存到 Google Drive。
下次训练时，Step 3 会自动检测并跳过生成步骤，直接进入训练。

---

**注意**：
- Colab 会话最长约 12 小时，训练期间请勿关闭浏览器
- 如果会话断开，重新运行整个 notebook 即可（辅助数据已在 Drive 中）